# MNIST 低成本 teacher / rollout 断层实验

这个 notebook 只做机制门控：比较普通 flow-matching MSE 与频带加权 MSE，检查 held-out teacher 指标改善是否会伴随自生成 rollout 变差。MNIST 结果不能替代 ImageNet/RAE 的最终生成结论。

## 1. Setup

数据固定从 `/data/shared/mnist` 读取；实验结果只在显式 `save=True` 时写到 `~/data/eqvae/experiments/`。

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'experiments').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.mnist_spectral_rollout_toy import (
    MNISTToyConfig, comparison_tables, plot_result, run_mnist_spectral_toy,
)

## 2. Simple Interface

`T(...)` 训练并评估一组成对模型；`V(result)` 汇总表格和图片。默认配置是单卡数分钟级。

In [ ]:
def T(seed=0, width=24, steps=1000, train_size=8192, test_size=1024,
      sample_count=1024, gamma=0.5, time_shift=1.0, save=False):
    config = MNISTToyConfig(
        seed=seed, width=width, steps=steps, train_size=train_size,
        test_size=test_size, sample_count=sample_count, gamma=gamma,
        time_shift=time_shift, device='cuda:0', save=save,
    )
    return run_mnist_spectral_toy(config)

def V(result):
    teacher_ratio, rollout_ratio = comparison_tables(result)
    display(teacher_ratio.style.format(precision=4).set_caption(
        'Teacher 指标：weighted / baseline，低于 1 表示加权更好'))
    display(rollout_ratio.style.format(precision=4).set_caption(
        'Rollout 分布：weighted / baseline，低于 1 表示加权更好；decoded 指标已裁剪到有效像素范围'))
    display(result.rollout_bands.pivot(index='band', columns='variant', values='log_energy_ratio')
            .style.format(precision=4).set_caption('生成/真实频带能量对数比，越接近 0 越好'))
    print(f'独立 MNIST test 分类准确率: {result.classifier_accuracy:.2%}')
    return plot_result(result)

## 3. Run

只需要修改下面这一行。先保持 `save=False`；需要保存 checkpoint 和 CSV 时再打开。

In [ ]:
R = T(seed=0, width=24, steps=1000, gamma=0.5, time_shift=1.0, save=False)

In [ ]:
V(R)

## 4. Capacity Check

最重要的后续对照是只改变 `width`。若小网络出现 teacher 改善但 rollout 恶化，而大网络缓解，才支持“加权损失改变有限容量分配”的机制。建议先比较 `width=16` 和 `width=48`，每个配置至少 3 个 seed。